# Notebook 4 — Locked Human + LLM analysis

This notebook implements the analysis hierarchy locked on **23 September 2026**, after the first five-model data collection was complete and Qwen 3.7 Plus had subsequently passed the pilot and before substantive inspection of the LLM results.

## Locked hierarchy
**Primary confirmatory question:** Do LLMs preserve the human income gradient in well-being inequality after accounting for each model's general tendency to compress human variation?

**Benchmarks:** human income gradient; mean-LS gradient; overall LLM under-dispersion.

**Supporting/prespecified:** within-country checks; RIF-variance regressions; adjacent-income RIF–Oaxaca diagnostic; alternative inequality measures.

**Exploratory:** distributional tails/quantiles and MAE consequences. Other socioeconomic dimensions are deliberately left for a later exploratory notebook.

## Primary normalization
For source \(m\) and income group \(j\),

\[
D_{jm}=\frac{SD_{jm}}{\sum_{j=1}^{10}w_jSD_{jm}},
\]

where \(w_j\) is the **fixed human sample share** in Q288 group \(j\), used for humans and every LLM. Under pure proportional compression, \(SD_{j,LLM}=cSD_{j,Human}\), the constant \(c\) cancels.

## Confirmatory inference
The principal test treats Q288 categorically and compares the full ten-category normalized profile for each LLM with the human profile using a **paired country-cluster bootstrap** and a global Wald test. A two-sided difference in the linear trend is reported only as a summary. No one-sided directional test is used.

## Protocol note: Qwen reinstatement and post-lock supporting analyses

Qwen 3.7 Plus was originally preregistered but was provisionally dropped after repeated API access failures. Before substantive inspection of the main LLM results, API access became functional and a 100-case Qwen 3.7 Plus pilot completed successfully. Qwen was therefore reinstated, returning the study to **six LLMs**.

The **primary confirmatory analysis in Section 4 is unchanged**: same normalization, fixed human income-group weights, paired country-cluster bootstrap, and global profile test.

After initial inspection of the aggregate normalized profiles, additional country-structure analyses were added in Section 5 as **supporting/post-lock analyses**. They do not replace or redefine the primary test. Section 5 now includes country-clustered FE analyses, country-specific normalized gradients, leave-one-country-out and influence diagnostics, respondent-bootstrap CIs for individual country gradients, a two-stage country + respondent hierarchical bootstrap for cross-country correspondence, approximate errors-in-variables sensitivity, and shrunken random-slope robustness.

No country is removed from the main analysis because of its observed result. Iraq, which is visually influential in the country-gradient plots, is retained in all main estimates and examined only in an explicitly labelled sensitivity analysis.

In [ ]:
# Run once in Deepnote if needed.
%pip install -q pandas numpy scipy statsmodels matplotlib adjustText

In [ ]:
from pathlib import Path
import hashlib, json, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from adjustText import adjust_text
from scipy import stats
import statsmodels.formula.api as smf
warnings.filterwarnings("ignore")

DATA_DIR=Path("output/full_wvs")
PROFILE_FILE=DATA_DIR/"profiles_for_prediction.csv"
PRED_DIR=DATA_DIR/"predictions"
OUTDIR=DATA_DIR/"analysis_locked"; OUTDIR.mkdir(parents=True,exist_ok=True)

# ------------------------------------------------------------------
# FIGURE OUTPUT POLICY
# ------------------------------------------------------------------
# Remove figure files left by earlier versions of Notebook 4 so that
# analysis_locked contains only figures generated by the current run.
# Numerical/tabular outputs (CSV files) are deliberately preserved.
for _old_fig in list(OUTDIR.glob("fig*")):
    if _old_fig.is_file() and _old_fig.suffix.lower() in {".pdf", ".pdf", ".svg", ".jpg", ".jpeg"}:
        _old_fig.unlink()

print("Cleared figure files from earlier Notebook 4 runs.")
print("Current notebook saves figures as vector PDF files only.")


EXPECTED_N=93901
EXPECTED_COUNTRIES=66
INCOME_GROUPS=np.arange(1,11)
BOOT_REPS=1000
BOOT_SEED=20260923
MIN_CELL_N=30

MODEL_FILES={
 "GPT-5.6 Luna":PRED_DIR/"openai_gpt56_luna_FULL.csv",
 "Claude Sonnet 5":PRED_DIR/"anthropic_sonnet5_FULL.csv",
 "Gemini 3.8 Flash":PRED_DIR/"google_gemini38_flash_FULL.csv",
 "DeepSeek V4.1 Flash":PRED_DIR/"deepseek_v41_flash_FULL.csv",
 "Gemma 4 31B":PRED_DIR/"deepinfra_gemma4_31b_FULL.csv",
 "Qwen 3.7 Plus":PRED_DIR/"alibaba_qwen37_plus_FULL.csv",
}
SOURCE_ORDER=["Human"]+list(MODEL_FILES)
SLUG={"Human":"human","GPT-5.6 Luna":"gpt56_luna","Claude Sonnet 5":"claude_sonnet5",
      "Gemini 3.8 Flash":"gemini38_flash","DeepSeek V4.1 Flash":"deepseek_v41_flash",
      "Gemma 4 31B":"gemma4_31b","Qwen 3.7 Plus":"qwen37_plus"}

plt.rcParams["figure.figsize"]=(8,5)
plt.rcParams["axes.spines.top"]=False
plt.rcParams["axes.spines.right"]=False

def sha256_file(path):
    h=hashlib.sha256()
    with open(path,"rb") as f:
        for chunk in iter(lambda:f.read(1024*1024),b""): h.update(chunk)
    return h.hexdigest()

## 1. Technical audit and frozen merged dataset
This must pass before substantive interpretation.

In [ ]:
if not PROFILE_FILE.exists(): raise FileNotFoundError(PROFILE_FILE)
base=pd.read_csv(PROFILE_FILE,low_memory=False)
req=["WVS_ROW_ID","Q49","Q288","B_COUNTRY_ALPHA","COUNTRY_NAME"]
miss=[c for c in req if c not in base]
if miss: raise ValueError(f"Missing profile columns: {miss}")

base["WVS_ROW_ID"]=pd.to_numeric(base.WVS_ROW_ID,errors="raise").astype(int)
base["Q49"]=pd.to_numeric(base.Q49,errors="raise")
base["Q288"]=pd.to_numeric(base.Q288,errors="raise").astype(int)
base["country"]=base.B_COUNTRY_ALPHA.astype(str)
base["country_name"]=base.COUNTRY_NAME.astype(str)

assert len(base)==EXPECTED_N
assert base.WVS_ROW_ID.is_unique
assert base.Q49.between(1,10).all()
assert base.Q288.between(1,10).all()
assert base.country.nunique()==EXPECTED_COUNTRIES
ids=set(base.WVS_ROW_ID)

audit=[{"source":"Human","file":str(PROFILE_FILE),"sha256":sha256_file(PROFILE_FILE),
        "n":len(base),"min":base.Q49.min(),"max":base.Q49.max()}]
pred_cols={}; hashes={}

for model,path in MODEL_FILES.items():
    if not path.exists(): raise FileNotFoundError(f"{model}: {path}")
    p=pd.read_csv(path,low_memory=False)
    if not {"WVS_ROW_ID","prediction"}.issubset(p): raise ValueError(f"{model}: missing ID/prediction")
    p["WVS_ROW_ID"]=pd.to_numeric(p.WVS_ROW_ID,errors="raise").astype(int)
    p["prediction"]=pd.to_numeric(p.prediction,errors="raise")
    assert len(p)==EXPECTED_N and p.WVS_ROW_ID.is_unique
    assert set(p.WVS_ROW_ID)==ids
    assert p.prediction.between(1,10).all()
    assert np.equal(p.prediction,np.floor(p.prediction)).all()
    if "status" in p: assert p.status.eq("ok").all()
    col="ls_"+SLUG[model]; pred_cols[model]=col
    base=base.merge(p[["WVS_ROW_ID","prediction"]].rename(columns={"prediction":col}),
                    on="WVS_ROW_ID",how="left",validate="one_to_one")
    base[col]=base[col].astype(int)
    hashes[model]=sha256_file(path)
    audit.append({"source":model,"file":str(path),"sha256":hashes[model],
                  "n":len(p),"min":p.prediction.min(),"max":p.prediction.max()})

support=["Q262","Q260","Q47","Q273","Q275","Q274","Q173"]

# Preserve an available WVS survey-weight variable for weighting robustness.
# S017 is the standard WVS integrated-data weight name; transparent
# alternatives are checked for compatibility with differently prepared files.
weight_candidates=["S017","W_WEIGHT","WEIGHT","weight"]
available_weight_cols=[c for c in weight_candidates if c in base.columns]

cols=(["WVS_ROW_ID","country","country_name","Q49","Q288"]
      +list(pred_cols.values())
      +[c for c in support if c in base]
      +available_weight_cols)
# Preserve order while removing accidental duplicates.
cols=list(dict.fromkeys(cols))
analysis=base[cols].copy()
pd.DataFrame(audit).to_csv(OUTDIR/"00_input_audit.csv",index=False)
analysis.to_csv(OUTDIR/"00_frozen_analysis_dataset.csv",index=False)

manifest={
 "locked_date":"2026-09-23","N":EXPECTED_N,"countries":EXPECTED_COUNTRIES,
 "profile_sha256":sha256_file(PROFILE_FILE),"prediction_sha256":hashes,
 "primary":"normalized SD profile across Q288",
 "normalization":"source-specific weighted mean of income-group SDs; fixed human Q288 weights",
 "inference":"paired country-cluster bootstrap; global profile Wald test; two-sided trend summary",
 "bootstrap_reps":BOOT_REPS,"bootstrap_seed":BOOT_SEED
}
json.dump(manifest,open(OUTDIR/"00_analysis_manifest.json","w"),indent=2)
display(pd.DataFrame(audit))
print("TECHNICAL AUDIT PASSED:",len(analysis),"rows;",analysis.country.nunique(),"countries/territories")

## 2. Common long-format data and descriptive moments

In [ ]:
source_to_col={"Human":"Q49",**pred_cols}
parts=[]
for s in SOURCE_ORDER:
    z=analysis[["WVS_ROW_ID","country","Q288"]].copy()
    z["source"]=s; z["ls"]=analysis[source_to_col[s]].astype(float)
    parts.append(z)
long=pd.concat(parts,ignore_index=True)
long["source"]=pd.Categorical(long.source,categories=SOURCE_ORDER,ordered=True)

def gini(x):
    x=np.sort(np.asarray(pd.Series(x).dropna(),float)); n=len(x)
    return (2*np.sum(np.arange(1,n+1)*x)/(n*np.sum(x)))-(n+1)/n if n and x.sum()>0 else np.nan

def gm(g):
    y=g.ls; mu=y.mean(); sd=y.std(ddof=1)
    return pd.Series({"n":len(y),"mean_ls":mu,"sd_ls":sd,"variance_ls":y.var(ddof=1),
      "median_ls":y.median(),"iqr_ls":y.quantile(.75)-y.quantile(.25),
      "cv_ls":sd/mu,"gini_ls":gini(y),"misery3":(y<=3).mean(),"low7":(y<=7).mean(),
      "high8":(y>=8).mean(),"max10":(y==10).mean(),"p10":y.quantile(.10),
      "p25":y.quantile(.25),"p50":y.quantile(.50),"p75":y.quantile(.75),"p90":y.quantile(.90)})

mom=(long.groupby(["source","Q288"],observed=True).apply(gm,include_groups=False).reset_index())
mom.to_csv(OUTDIR/"01_income_group_moments_all_sources.csv",index=False)
display(mom.round(4))

In [ ]:
for metric,ylabel,title,fname in [
 ("mean_ls","Mean life satisfaction","Mean life satisfaction by income group","fig01_mean_ls_by_income.pdf"),
 ("sd_ls","SD of life satisfaction","Raw well-being dispersion by income group","fig02_raw_sd_by_income.pdf")]:
    plt.figure(figsize=(9,6))
    for s in SOURCE_ORDER:
        g=mom[mom.source.eq(s)]
        plt.plot(g.Q288,g[metric],marker="o",label=s)
    plt.xlabel("WVS income-group position (Q288: 1 lowest → 10 highest)")
    plt.ylabel(ylabel); plt.title(title); plt.xticks(INCOME_GROUPS); plt.legend(fontsize=8)
    plt.tight_layout(); plt.savefig(OUTDIR/fname,bbox_inches="tight"); plt.show()

## 3. General compression benchmark
Report overall SD and the human-weighted mean of within-income SDs. The latter is the scale removed by the primary normalization.

In [ ]:
counts=analysis.groupby("Q288").size().reindex(INCOME_GROUPS).astype(float)
w=(counts/counts.sum()).values
rows=[]; scales={}
for s in SOURCE_ORDER:
    y=long.loc[long.source.eq(s),"ls"]
    g=mom.loc[mom.source.eq(s)].set_index("Q288").reindex(INCOME_GROUPS)
    scale=float(np.sum(w*g.sd_ls.values)); scales[s]=scale
    rows.append({"source":s,"overall_mean":y.mean(),"overall_sd":y.std(ddof=1),
                 "weighted_mean_within_income_sd":scale})
compression=pd.DataFrame(rows)
h0=compression.loc[compression.source.eq("Human")].iloc[0]
compression["overall_sd_ratio_vs_human"]=compression.overall_sd/h0.overall_sd
compression["within_income_scale_ratio_vs_human"]=compression.weighted_mean_within_income_sd/h0.weighted_mean_within_income_sd
compression.to_csv(OUTDIR/"02_general_compression.csv",index=False)
display(compression.round(4))

# 4. PRIMARY CONFIRMATORY ANALYSIS

This section contains the paper's **primary confirmatory evidence**. The central figure is the normalized income–well-being inequality profile. Normalization removes each source's overall dispersion scale using the same fixed human income-group weights, allowing the analysis to test whether LLMs preserve the *structure* of the human income gradient in well-being inequality despite general under-dispersion.

**Primary figure produced below:** `fig03_PRIMARY_normalized_sd_profiles.pdf`

All country-specific analyses introduced after the analysis lock appear later as supporting/post-lock analyses and do not replace this primary test.


In [ ]:
rows=[]
for s in SOURCE_ORDER:
    g=mom.loc[mom.source.eq(s)].set_index("Q288").reindex(INCOME_GROUPS)
    for q in INCOME_GROUPS:
        rows.append({"source":s,"Q288":q,"sd_ls":g.loc[q,"sd_ls"],
                     "scale":scales[s],"normalized_sd":g.loc[q,"sd_ls"]/scales[s]})
norm=pd.DataFrame(rows)
norm.to_csv(OUTDIR/"03_primary_normalized_sd_profiles.csv",index=False)
for s in SOURCE_ORDER:
    v=norm.loc[norm.source.eq(s)].set_index("Q288").reindex(INCOME_GROUPS).normalized_sd.values
    assert np.isclose(np.sum(w*v),1)

plt.figure(figsize=(9,6))
for s in SOURCE_ORDER:
    g=norm[norm.source.eq(s)]
    plt.plot(g.Q288,g.normalized_sd,marker="o",label=s)
plt.axhline(1,linewidth=1,linestyle=":")
plt.xlabel("WVS income-group position (Q288: 1 lowest → 10 highest)")
plt.ylabel("Normalized SD of life satisfaction")
plt.title("PRIMARY CONFIRMATORY RESULT: Normalized income–well-being inequality profiles")
plt.xticks(INCOME_GROUPS); plt.legend(fontsize=8); plt.tight_layout()
plt.savefig(OUTDIR/"fig03_PRIMARY_normalized_sd_profiles.pdf",bbox_inches="tight"); plt.show()

## 4A. Paired country-cluster bootstrap
Resample countries with replacement. Fixed original human income weights are retained in every replication.

In [ ]:
countries=np.array(sorted(analysis.country.unique()))
C0=len(countries); J=10; M0=len(SOURCE_ORDER)
ci={c:i for i,c in enumerate(countries)}; qi={q:i for i,q in enumerate(INCOME_GROUPS)}
nA=np.zeros((C0,J,M0)); sA=np.zeros_like(nA); ssA=np.zeros_like(nA)

for m,s in enumerate(SOURCE_ORDER):
    col=source_to_col[s]
    a=(analysis.groupby(["country","Q288"])[col]
       .agg(["count","sum",lambda x:np.sum(np.square(x))]).reset_index())
    a.columns=["country","Q288","n","sum","sumsq"]
    for r in a.itertuples(index=False):
        i,j=ci[r.country],qi[int(r.Q288)]
        nA[i,j,m],sA[i,j,m],ssA[i,j,m]=r.n,r.sum,r.sumsq

def sd_suff(n,s,ss):
    v=(ss-s*s/n)/(n-1)
    return np.sqrt(np.maximum(v,0))

obs_sd=sd_suff(nA.sum(0),sA.sum(0),ssA.sum(0))
for m,s in enumerate(SOURCE_ORDER):
    direct=mom.loc[mom.source.eq(s)].set_index("Q288").reindex(INCOME_GROUPS).sd_ls.values
    assert np.allclose(obs_sd[:,m],direct,atol=1e-10)

obs_scale=np.sum(w[:,None]*obs_sd,axis=0)
obs_norm=obs_sd/obs_scale[None,:]

x=INCOME_GROUPS.astype(float); xc=x-x.mean(); den=np.sum(xc**2)
obs_trend=np.sum(xc[:,None]*obs_norm,axis=0)/den

rng=np.random.default_rng(BOOT_SEED)
boot_norm=np.empty((BOOT_REPS,J,M0)); boot_trend=np.empty((BOOT_REPS,M0))
for b in range(BOOT_REPS):
    draw=rng.integers(0,C0,size=C0)
    bn,bs,bss=nA[draw].sum(0),sA[draw].sum(0),ssA[draw].sum(0)
    sd=sd_suff(bn,bs,bss); sc=np.sum(w[:,None]*sd,axis=0); z=sd/sc[None,:]
    boot_norm[b]=z; boot_trend[b]=np.sum(xc[:,None]*z,axis=0)/den
print("Bootstrap complete:",BOOT_REPS,"country-cluster replications")

## 4B. Confirmatory tests
The global ten-category profile test is primary. The linear trend difference is a two-sided summary.

In [ ]:
idx={s:i for i,s in enumerate(SOURCE_ORDER)}; hm=idx["Human"]; tests=[]
for model in SOURCE_ORDER[1:]:
    m=idx[model]
    d=obs_norm[:,m]-obs_norm[:,hm]
    db=boot_norm[:,:,m]-boot_norm[:,:,hm]
    V=np.cov(db,rowvar=False,ddof=1); rank=int(np.linalg.matrix_rank(V))
    W=float(d.T@np.linalg.pinv(V)@d); pg=float(stats.chi2.sf(W,df=rank))
    td=obs_trend[m]-obs_trend[hm]; tdb=boot_trend[:,m]-boot_trend[:,hm]
    lo,hi=np.quantile(tdb,[.025,.975])
    centered=tdb-tdb.mean()
    pt=min(1.0,2*min(np.mean(centered<=-abs(td)),np.mean(centered>=abs(td))))
    tests.append({"model":model,"global_wald_chi2":W,"global_wald_df_rank":rank,
      "global_wald_p":pg,"human_linear_trend":obs_trend[hm],"model_linear_trend":obs_trend[m],
      "trend_difference_model_minus_human":td,"trend_ci_low":lo,"trend_ci_high":hi,
      "trend_p_two_sided":pt,"profile_rmsd_vs_human":np.sqrt(np.mean(d*d))})
primary=pd.DataFrame(tests)
primary.to_csv(OUTDIR/"04_PRIMARY_confirmatory_tests.csv",index=False)
display(primary.round(5))

cir=[]
for s in SOURCE_ORDER:
    m=idx[s]
    for j,q in enumerate(INCOME_GROUPS):
        lo,hi=np.quantile(boot_norm[:,j,m],[.025,.975])
        cir.append({"source":s,"Q288":q,"normalized_sd":obs_norm[j,m],"ci_low":lo,"ci_high":hi})
pd.DataFrame(cir).to_csv(OUTDIR/"05_primary_normalized_profiles_bootstrap_ci.csv",index=False)

## 4C. ROBUSTNESS: weighting sensitivity

This section checks whether the primary normalized income-dispersion profile is sensitive to how countries/respondents are weighted.

- **Equal-country weighting:** each respondent receives weight \(1/N_c\), so every country contributes the same total weight.
- **WVS survey weighting:** if a standard WVS survey-weight variable is available in `profiles_for_prediction.csv`, the same respondent weights are applied to the Human outcome and all six LLM outcomes.

For each weighting scheme, income-group shares are calculated from the weighted **human** sample and then held fixed across Human and all LLM sources, preserving the logic of the primary normalization. These are robustness analyses; the preregistered unweighted analysis remains primary.


In [ ]:
# ============================================================
# ROBUSTNESS: EQUAL-COUNTRY AND WVS SURVEY WEIGHTING
# ============================================================

def _weighted_mean_sd(y, wt):
    y = np.asarray(y, dtype=float)
    wt = np.asarray(wt, dtype=float)
    ok = np.isfinite(y) & np.isfinite(wt) & (wt > 0)
    y, wt = y[ok], wt[ok]
    if len(y) < 2 or wt.sum() <= 0:
        return np.nan, np.nan, 0.0
    sw = wt.sum()
    mu = np.sum(wt * y) / sw
    # Probability-weighted SD; sufficient for this sensitivity analysis.
    var = np.sum(wt * (y - mu) ** 2) / sw
    return float(mu), float(np.sqrt(max(var, 0.0))), float(sw)

# Every country contributes the same total respondent weight.
country_n = analysis.groupby("country")["WVS_ROW_ID"].transform("size").astype(float)
analysis["_wt_equal_country"] = 1.0 / country_n

weight_schemes = {"equal_country": "_wt_equal_country"}

# WVS Wave 7 commonly uses S017 as the sampling/post-stratification weight.
# The code also checks a small set of transparent alternative names.
weight_candidates = ["W_WEIGHT", "PWGHT", "S017", "WEIGHT", "weight"]
survey_weight_col = next((c for c in weight_candidates if c in analysis.columns), None)

weight_audit = [{
    "scheme": "equal_country",
    "available": True,
    "source_column": "_wt_equal_country",
    "n_positive": int((analysis["_wt_equal_country"] > 0).sum()),
    "n_missing": int(analysis["_wt_equal_country"].isna().sum())
}]

if survey_weight_col is not None:
    analysis["_wt_wvs_survey"] = pd.to_numeric(analysis[survey_weight_col], errors="coerce")
    analysis.loc[~np.isfinite(analysis["_wt_wvs_survey"]) | (analysis["_wt_wvs_survey"] <= 0),
                 "_wt_wvs_survey"] = np.nan
    npos = int(analysis["_wt_wvs_survey"].notna().sum())
    if npos > 0:
        weight_schemes["wvs_survey"] = "_wt_wvs_survey"
        weight_audit.append({
            "scheme": "wvs_survey",
            "available": True,
            "source_column": survey_weight_col,
            "n_positive": npos,
            "n_missing": int(analysis["_wt_wvs_survey"].isna().sum())
        })
    else:
        weight_audit.append({
            "scheme": "wvs_survey",
            "available": False,
            "source_column": survey_weight_col,
            "n_positive": 0,
            "n_missing": len(analysis)
        })
else:
    weight_audit.append({
        "scheme": "wvs_survey",
        "available": False,
        "source_column": "",
        "n_positive": 0,
        "n_missing": len(analysis)
    })

weight_audit = pd.DataFrame(weight_audit)
weight_audit.to_csv(OUTDIR / "04a_weighting_robustness_audit.csv", index=False)
display(weight_audit)

wr_profiles = []
wr_summary = []

primary_lookup = (norm.set_index(["source", "Q288"])["normalized_sd"].to_dict())
primary_human = np.array([primary_lookup[("Human", q)] for q in INCOME_GROUPS])

for scheme, wtcol in weight_schemes.items():
    # Scheme-specific HUMAN income shares, then held fixed across all sources.
    q_mass = (analysis.groupby("Q288")[wtcol].sum()
              .reindex(INCOME_GROUPS).astype(float))
    q_weights = (q_mass / q_mass.sum()).to_numpy()

    scheme_profiles = {}

    for s in SOURCE_ORDER:
        col = source_to_col[s]
        sd_by_q = []
        mean_by_q = []

        for q in INCOME_GROUPS:
            mask = analysis["Q288"].eq(q)
            mu_q, sd_q, sw_q = _weighted_mean_sd(
                analysis.loc[mask, col],
                analysis.loc[mask, wtcol]
            )
            mean_by_q.append(mu_q)
            sd_by_q.append(sd_q)

        sd_by_q = np.asarray(sd_by_q, float)
        scale = float(np.sum(q_weights * sd_by_q))
        normalized = sd_by_q / scale
        scheme_profiles[s] = normalized

        for j, q in enumerate(INCOME_GROUPS):
            wr_profiles.append({
                "weighting_scheme": scheme,
                "source": s,
                "Q288": int(q),
                "weighted_mean_ls": mean_by_q[j],
                "weighted_sd_ls": sd_by_q[j],
                "human_income_share_weight": q_weights[j],
                "normalization_scale": scale,
                "normalized_sd": normalized[j]
            })

    human_prof = scheme_profiles["Human"]
    human_trend = float(np.sum(xc * human_prof) / den)

    for s in SOURCE_ORDER:
        prof = scheme_profiles[s]
        trend = float(np.sum(xc * prof) / den)
        primary_prof = np.array([primary_lookup[(s, q)] for q in INCOME_GROUPS])
        wr_summary.append({
            "weighting_scheme": scheme,
            "source": s,
            "normalized_linear_trend": trend,
            "trend_difference_model_minus_human":
                np.nan if s == "Human" else trend - human_trend,
            "profile_rmsd_vs_human":
                0.0 if s == "Human" else float(np.sqrt(np.mean((prof - human_prof) ** 2))),
            "profile_rmsd_vs_primary_same_source":
                float(np.sqrt(np.mean((prof - primary_prof) ** 2)))
        })

weighting_profiles = pd.DataFrame(wr_profiles)
weighting_summary = pd.DataFrame(wr_summary)

weighting_profiles.to_csv(
    OUTDIR / "04b_weighting_robustness_normalized_profiles.csv", index=False
)
weighting_summary.to_csv(
    OUTDIR / "04c_weighting_robustness_summary.csv", index=False
)

print("Weighting robustness schemes completed:", ", ".join(weight_schemes))
if "wvs_survey" not in weight_schemes:
    print(
        "NOTE: No usable WVS survey-weight column was found in profiles_for_prediction.csv. "
        "Equal-country weighting was completed. If survey weighting is required, carry S017 "
        "(or the applicable WVS weight variable) into the prepared profile file and rerun."
    )

display(weighting_summary.round(5))


## 5. SUPPORTING / POST-LOCK: Within-country robustness and country-level gradient fidelity

These analyses are downstream of the locked primary confirmatory test in Section 4. They examine whether the normalized income–dispersion relationship survives country adjustment and whether cross-country heterogeneity in the human gradient is reproduced by each LLM. They do **not** redefine or replace the primary estimand.


In [ ]:
# ------------------------------------------------------------
# 5A. Country × income cells and conventional country-FE slopes
# ------------------------------------------------------------
cells=[]
for s in SOURCE_ORDER:
    col=source_to_col[s]
    cc=(analysis.groupby(["country","country_name","Q288"])[col]
        .agg(n="size",mean_ls="mean",sd_ls="std",variance_ls="var").reset_index())
    cc=cc[cc.n>=MIN_CELL_N].copy()
    cc["source"]=s
    cells.append(cc)

ccall=pd.concat(cells,ignore_index=True)
ccall.to_csv(OUTDIR/"06_country_income_cells_all_sources.csv",index=False)

fer=[]; csr=[]
for s in SOURCE_ORDER:
    cc=ccall[ccall.source.eq(s)].copy()

    for y in ["mean_ls","sd_ls","variance_ls"]:
        for wt in ["N-weighted","equal-cell"]:
            if wt=="N-weighted":
                fit=smf.wls(f"{y} ~ Q288 + C(country)",cc,weights=cc.n).fit(
                    cov_type="cluster",cov_kwds={"groups":cc.country})
            else:
                fit=smf.ols(f"{y} ~ Q288 + C(country)",cc).fit(
                    cov_type="cluster",cov_kwds={"groups":cc.country})
            fer.append({"source":s,"outcome":y,"weighting":wt,
                        "slope":fit.params["Q288"],"se":fit.bse["Q288"],
                        "p":fit.pvalues["Q288"]})

    for (country,country_name),g in cc.groupby(["country","country_name"]):
        if g.Q288.nunique()<5:
            continue
        for y in ["mean_ls","sd_ls","variance_ls"]:
            f=smf.wls(f"{y} ~ Q288",g,weights=g.n).fit()
            csr.append({"source":s,"country":country,"country_name":country_name,
                        "outcome":y,"n_income_cells":g.Q288.nunique(),
                        "slope":f.params["Q288"]})

fe_slopes=pd.DataFrame(fer)
country_raw_slopes=pd.DataFrame(csr)
fe_slopes.to_csv(OUTDIR/"07_within_country_FE_slopes.csv",index=False)
country_raw_slopes.to_csv(OUTDIR/"08_country_specific_raw_slopes.csv",index=False)
display(fe_slopes.round(5))

# -------------------------------------------------------------------
# 5B. Country-FE analogue of the primary scale normalization
# -------------------------------------------------------------------
# Fixed HUMAN cell Ns/shares across all sources.
human_cc=(ccall[ccall.source.eq("Human")]
          [["country","country_name","Q288","n"]]
          .rename(columns={"n":"human_n"}))

ccn=ccall.merge(human_cc[["country","Q288","human_n"]],
                on=["country","Q288"],how="left",validate="many_to_one")

# Remove each source's general dispersion scale using fixed human weights
# over retained country×income cells.
global_human_weights=(human_cc.assign(w=lambda x:x.human_n/x.human_n.sum())
                      [["country","Q288","w"]])
ccn=ccn.merge(global_human_weights,on=["country","Q288"],how="left",validate="many_to_one")
global_scale=(ccn.assign(wsd=lambda x:x.w*x.sd_ls)
              .groupby("source",as_index=False).wsd.sum()
              .rename(columns={"wsd":"global_sd_scale"}))
ccn=ccn.merge(global_scale,on="source",how="left",validate="many_to_one")
ccn["sd_norm_global"]=ccn.sd_ls/ccn.global_sd_scale

# Linear FE summary, with country-clustered SEs.
norm_fe=[]
for s in SOURCE_ORDER:
    cc=ccn[ccn.source.eq(s)].copy()
    for wt in ["N-weighted","equal-cell"]:
        if wt=="N-weighted":
            fit=smf.wls("sd_norm_global ~ Q288 + C(country)",cc,weights=cc.human_n).fit(
                cov_type="cluster",cov_kwds={"groups":cc.country})
        else:
            fit=smf.ols("sd_norm_global ~ Q288 + C(country)",cc).fit(
                cov_type="cluster",cov_kwds={"groups":cc.country})
        norm_fe.append({"source":s,"weighting":wt,
                        "normalized_income_slope":fit.params["Q288"],
                        "se":fit.bse["Q288"],"p":fit.pvalues["Q288"]})

norm_fe=pd.DataFrame(norm_fe)
norm_fe.to_csv(OUTDIR/"08a_country_FE_normalized_sd_slopes.csv",index=False)
display(norm_fe.round(5))

# Categorical FE profile: preserves nonlinearity rather than forcing a single slope.
norm_fe_cat=[]
for s in SOURCE_ORDER:
    cc=ccn[ccn.source.eq(s)].copy()
    fit=smf.wls("sd_norm_global ~ C(Q288) + C(country)",cc,weights=cc.human_n).fit(
        cov_type="cluster",cov_kwds={"groups":cc.country})
    for q in INCOME_GROUPS:
        term=f"C(Q288)[T.{q}]"
        norm_fe_cat.append({
            "source":s,"Q288":int(q),
            "coef_vs_1":0.0 if q==1 else fit.params.get(term,np.nan),
            "se":0.0 if q==1 else fit.bse.get(term,np.nan),
            "p":np.nan if q==1 else fit.pvalues.get(term,np.nan)
        })

norm_fe_cat=pd.DataFrame(norm_fe_cat)
norm_fe_cat.to_csv(OUTDIR/"08a2_country_FE_normalized_sd_categorical.csv",index=False)

# -------------------------------------------------------------------
# 5C. Country-specific NORMALIZED income–well-being inequality gradients
# -------------------------------------------------------------------
# Normalize WITHIN country and source. Denominators use fixed HUMAN Q288
# shares among baseline retained cells in that country.
country_human_weights=(human_cc.assign(
    cw=lambda x:x.human_n/x.groupby("country").human_n.transform("sum"))
    [["country","Q288","cw"]])
ccn=ccn.merge(country_human_weights,on=["country","Q288"],how="left",validate="many_to_one")

country_scales=(ccn.assign(cwsd=lambda x:x.cw*x.sd_ls)
                .groupby(["source","country"],as_index=False).cwsd.sum()
                .rename(columns={"cwsd":"country_sd_scale"}))
ccn=ccn.merge(country_scales,on=["source","country"],how="left",validate="many_to_one")
ccn["sd_norm_country"]=ccn.sd_ls/ccn.country_sd_scale
ccn.to_csv(OUTDIR/"08b_country_income_cells_normalized.csv",index=False)

cgrad=[]
for s in SOURCE_ORDER:
    cc=ccn[ccn.source.eq(s)].copy()
    for (country,country_name),g in cc.groupby(["country","country_name"]):
        if g.Q288.nunique()<5:
            continue
        f=smf.wls("sd_norm_country ~ Q288",g,weights=g.human_n).fit()
        cgrad.append({
            "source":s,"country":country,"country_name":country_name,
            "n_income_cells":int(g.Q288.nunique()),
            "normalized_gradient":f.params["Q288"],
            "se_descriptive":f.bse["Q288"],
            "r2":f.rsquared
        })

country_gradients=pd.DataFrame(cgrad)
country_gradients.to_csv(OUTDIR/"08c_country_specific_normalized_sd_gradients.csv",index=False)

grad_wide=(country_gradients
           .pivot_table(index=["country","country_name"],columns="source",
                        values="normalized_gradient")
           .reset_index())
grad_wide.to_csv(OUTDIR/"08d_country_gradient_wide.csv",index=False)

# Descriptive cross-country correspondence and 45-degree distance.
corr_rows=[]
for model in SOURCE_ORDER[1:]:
    d=grad_wide[["country","country_name","Human",model]].dropna().copy()
    pearson_r,pearson_p=stats.pearsonr(d["Human"],d[model])
    spearman_r,spearman_p=stats.spearmanr(d["Human"],d[model])
    reg=smf.ols(f'Q("{model}") ~ Human',d).fit()
    diff=d[model]-d["Human"]

    corr_rows.append({
        "model":model,"n_countries":len(d),
        "pearson_r":pearson_r,"pearson_p_naive":pearson_p,
        "spearman_rho":spearman_r,"spearman_p_naive":spearman_p,
        "ols_intercept":reg.params["Intercept"],
        "ols_slope":reg.params["Human"],"r2":reg.rsquared,
        "sign_agreement":np.mean(np.sign(d["Human"])==np.sign(d[model])),
        "identity_bias_mean":diff.mean(),
        "identity_mae":np.abs(diff).mean(),
        "identity_rmsd":np.sqrt(np.mean(diff**2))
    })

gradient_correspondence=pd.DataFrame(corr_rows)
gradient_correspondence.to_csv(OUTDIR/"08e_country_gradient_correspondence_descriptive.csv",index=False)
display(gradient_correspondence.round(4))

# -------------------------------------------------------------------
# 5D. Influence, leave-one-country-out, and Iraq sensitivity
# -------------------------------------------------------------------
loo_rows=[]; influence_rows=[]; sensitivity_rows=[]
for model in SOURCE_ORDER[1:]:
    d=grad_wide[["country","country_name","Human",model]].dropna().copy()
    fit=smf.ols(f'Q("{model}") ~ Human',d).fit()
    inf=fit.get_influence()
    d["cooks_d"]=inf.cooks_distance[0]
    d["studentized_resid"]=inf.resid_studentized_external
    d["leverage"]=inf.hat_matrix_diag
    for _,r in d.iterrows():
        influence_rows.append({"model":model,"country":r.country,"country_name":r.country_name,
                               "cooks_d":r.cooks_d,"studentized_resid":r.studentized_resid,
                               "leverage":r.leverage})
        dl=d[d.country.ne(r.country)]
        lf=smf.ols(f'Q("{model}") ~ Human',dl).fit()
        loo_rows.append({"model":model,"country_dropped":r.country,
                         "country_name_dropped":r.country_name,
                         "pearson_r":stats.pearsonr(dl.Human,dl[model]).statistic,
                         "spearman_rho":stats.spearmanr(dl.Human,dl[model]).statistic,
                         "ols_slope":lf.params["Human"],"r2":lf.rsquared})
    no_irq=d[~d.country_name.str.casefold().eq("iraq")].copy()
    if len(no_irq)<len(d):
        sf=smf.ols(f'Q("{model}") ~ Human',no_irq).fit()
        sensitivity_rows.append({"model":model,"sample":"Excluding Iraq","n_countries":len(no_irq),
            "pearson_r":stats.pearsonr(no_irq.Human,no_irq[model]).statistic,
            "spearman_rho":stats.spearmanr(no_irq.Human,no_irq[model]).statistic,
            "ols_intercept":sf.params["Intercept"],"ols_slope":sf.params["Human"],"r2":sf.rsquared,
            "identity_mae":np.mean(np.abs(no_irq[model]-no_irq.Human)),
            "identity_rmsd":np.sqrt(np.mean((no_irq[model]-no_irq.Human)**2))})

loo=pd.DataFrame(loo_rows); influence=pd.DataFrame(influence_rows)
iraq_sensitivity=pd.DataFrame(sensitivity_rows)
loo.to_csv(OUTDIR/"08f_leave_one_country_out.csv",index=False)
influence.to_csv(OUTDIR/"08f2_country_influence_diagnostics.csv",index=False)
iraq_sensitivity.to_csv(OUTDIR/"08f3_excluding_iraq_sensitivity.csv",index=False)
loo_summary=(loo.groupby("model").agg(
    pearson_min=("pearson_r","min"),pearson_max=("pearson_r","max"),
    spearman_min=("spearman_rho","min"),spearman_max=("spearman_rho","max"),
    slope_min=("ols_slope","min"),slope_max=("ols_slope","max")).reset_index())
loo_summary.to_csv(OUTDIR/"08f4_leave_one_country_out_summary.csv",index=False)
display(loo_summary.round(4)); display(iraq_sensitivity.round(4))

# -------------------------------------------------------------------
# 5E. Paired respondent bootstrap WITHIN countries
# -------------------------------------------------------------------
COUNTRY_BOOT_REPS=1000; COUNTRY_BOOT_SEED=20260924
rng_country=np.random.default_rng(COUNTRY_BOOT_SEED)
Y=analysis[[source_to_col[s] for s in SOURCE_ORDER]].to_numpy(float)
q_arr=analysis.Q288.to_numpy(int); country_arr=analysis.country.to_numpy()
country_name_map=(analysis[["country","country_name"]].drop_duplicates()
                  .set_index("country").country_name.to_dict())
base_cells=human_cc[["country","Q288","human_n"]].copy()
eligible_countries=(base_cells.groupby("country").Q288.nunique().loc[lambda x:x>=5].index.tolist())
base_cells=base_cells[base_cells.country.isin(eligible_countries)].copy()
base_cells["cw"]=base_cells.human_n/base_cells.groupby("country").human_n.transform("sum")
cell_key={(r.country,int(r.Q288)):(float(r.human_n),float(r.cw)) for r in base_cells.itertuples(index=False)}
country_indices={c:np.flatnonzero(country_arr==c) for c in eligible_countries}

def weighted_slope(x,y,w):
    x=np.asarray(x,float); y=np.asarray(y,float); w=np.asarray(w,float)
    ok=np.isfinite(x)&np.isfinite(y)&np.isfinite(w)&(w>0); x=x[ok]; y=y[ok]; w=w[ok]
    if len(x)<2 or np.ptp(x)==0: return np.nan
    xb=np.average(x,weights=w); yb=np.average(y,weights=w); den=np.sum(w*(x-xb)**2)
    return np.sum(w*(x-xb)*(y-yb))/den if den>0 else np.nan

def one_country_gradients(c,rng):
    idx=country_indices[c]; samp=idx[rng.integers(0,len(idx),size=len(idx))]
    qb=q_arr[samp]; yb=Y[samp,:]; qs=[]; ns=[]; cws=[]; sds=[]
    for q in sorted(base_cells.loc[base_cells.country.eq(c),"Q288"].astype(int).unique()):
        m=(qb==q)
        if m.sum()<2: continue
        n0,cw0=cell_key[(c,q)]; qs.append(q); ns.append(n0); cws.append(cw0)
        sds.append(np.std(yb[m,:],axis=0,ddof=1))
    if len(qs)<5: return None
    qs=np.asarray(qs,float); ns=np.asarray(ns,float); cws=np.asarray(cws,float); sds=np.vstack(sds)
    cws=cws/cws.sum(); scales=np.sum(cws[:,None]*sds,axis=0); norm=sds/scales[None,:]
    return np.array([weighted_slope(qs,norm[:,k],ns) for k in range(len(SOURCE_ORDER))])

boot_country=[]
for b in range(COUNTRY_BOOT_REPS):
    for c in eligible_countries:
        slopes=one_country_gradients(c,rng_country)
        if slopes is None: continue
        for k,s in enumerate(SOURCE_ORDER): boot_country.append((b,c,s,slopes[k]))
boot_country=pd.DataFrame(boot_country,columns=["rep","country","source","gradient"])
country_ci=(boot_country.groupby(["country","source"]).gradient.agg(
    boot_mean="mean",boot_se="std",ci_low=lambda x:np.nanquantile(x,.025),
    ci_high=lambda x:np.nanquantile(x,.975)).reset_index())
country_ci["country_name"]=country_ci.country.map(country_name_map)
country_ci.to_csv(OUTDIR/"08g_country_gradient_respondent_bootstrap_CI.csv",index=False)

# -------------------------------------------------------------------
# 5F. Two-stage hierarchical bootstrap: countries + respondents
# -------------------------------------------------------------------
HIER_BOOT_REPS=1000; HIER_BOOT_SEED=20260925
rng_hier=np.random.default_rng(HIER_BOOT_SEED); hier_rows=[]
for b in range(HIER_BOOT_REPS):
    draws=rng_hier.choice(eligible_countries,size=len(eligible_countries),replace=True)
    mat=[]
    for c in draws:
        slopes=one_country_gradients(c,rng_hier)
        if slopes is not None and np.all(np.isfinite(slopes)): mat.append(slopes)
    if len(mat)<3: continue
    mat=np.vstack(mat); h=mat[:,0]
    for k,model in enumerate(SOURCE_ORDER[1:],start=1):
        y=mat[:,k]; X=np.column_stack([np.ones(len(h)),h]); intercept,slope=np.linalg.lstsq(X,y,rcond=None)[0]
        yhat=X@np.array([intercept,slope]); sst=np.sum((y-y.mean())**2); diff=y-h
        hier_rows.append({"rep":b,"model":model,"n_country_draws":len(h),
            "pearson_r":stats.pearsonr(h,y).statistic,"spearman_rho":stats.spearmanr(h,y).statistic,
            "ols_intercept":intercept,"ols_slope":slope,"r2":1-np.sum((y-yhat)**2)/sst if sst>0 else np.nan,
            "sign_agreement":np.mean(np.sign(h)==np.sign(y)),"identity_bias_mean":np.mean(diff),
            "identity_mae":np.mean(np.abs(diff)),"identity_rmsd":np.sqrt(np.mean(diff**2))})
hier_boot=pd.DataFrame(hier_rows)
hier_boot.to_csv(OUTDIR/"08h_hierarchical_country_respondent_bootstrap_replicates.csv",index=False)
metrics=["pearson_r","spearman_rho","ols_intercept","ols_slope","r2","sign_agreement","identity_bias_mean","identity_mae","identity_rmsd"]
summary=[]
for model in SOURCE_ORDER[1:]:
    point=gradient_correspondence[gradient_correspondence.model.eq(model)].iloc[0]; bm=hier_boot[hier_boot.model.eq(model)]
    row={"model":model,"n_countries":int(point.n_countries),"bootstrap_reps":int(bm.rep.nunique())}
    for metric in metrics:
        row[f"{metric}_point"]=point[metric]; row[f"{metric}_boot_se"]=bm[metric].std(ddof=1)
        row[f"{metric}_ci_low"]=bm[metric].quantile(.025); row[f"{metric}_ci_high"]=bm[metric].quantile(.975)
    summary.append(row)
hier_summary=pd.DataFrame(summary)
hier_summary.to_csv(OUTDIR/"08i_country_gradient_correspondence_HIERARCHICAL_bootstrap_CI.csv",index=False)
display(hier_summary.round(4))

# -------------------------------------------------------------------
# 5G. Approximate errors-in-variables sensitivity
# -------------------------------------------------------------------
rel=[]
for s in SOURCE_ORDER:
    p=country_gradients[country_gradients.source.eq(s)].set_index("country").normalized_gradient
    se=country_ci[country_ci.source.eq(s)].set_index("country").boot_se; common=p.index.intersection(se.index)
    ov=float(np.var(p.loc[common],ddof=1)); sv=float(np.mean(se.loc[common]**2))
    rr=max(0,min(1,1-sv/ov)) if ov>0 else np.nan
    rel.append({"source":s,"observed_between_country_variance":ov,"mean_bootstrap_sampling_variance":sv,"approx_reliability":rr})
reliability=pd.DataFrame(rel); reliability.to_csv(OUTDIR/"08j_country_gradient_reliability.csv",index=False)
rm=reliability.set_index("source").approx_reliability.to_dict(); disatt=[]
for model in SOURCE_ORDER[1:]:
    r=float(gradient_correspondence.loc[gradient_correspondence.model.eq(model),"pearson_r"].iloc[0])
    den=np.sqrt(rm.get("Human",np.nan)*rm.get(model,np.nan)); rc=r/den if np.isfinite(den) and den>0 else np.nan
    if np.isfinite(rc): rc=float(np.clip(rc,-1,1))
    disatt.append({"model":model,"pearson_r_observed":r,"human_reliability":rm.get("Human",np.nan),
                   "model_reliability":rm.get(model,np.nan),"pearson_r_disattenuated_sensitivity":rc})
disatt=pd.DataFrame(disatt); disatt.to_csv(OUTDIR/"08k_country_gradient_disattenuated_correlation_sensitivity.csv",index=False)
display(disatt.round(4))

# -------------------------------------------------------------------
# 5H. Random-slope shrinkage robustness
# -------------------------------------------------------------------
shrunk=[]
for s in SOURCE_ORDER:
    cc=ccn[ccn.source.eq(s)].copy()
    try:
        mf=smf.mixedlm("sd_norm_country ~ Q288",cc,groups=cc.country,re_formula="~Q288").fit(reml=True,method="lbfgs",maxiter=1000,disp=False)
        fixed=float(mf.fe_params["Q288"])
        for c,re in mf.random_effects.items():
            rs=float(re["Q288"]) if "Q288" in re.index else np.nan
            shrunk.append({"source":s,"country":c,"shrunk_gradient":fixed+rs,"fixed_gradient":fixed,
                           "random_slope":rs,"converged":bool(mf.converged)})
    except Exception as e: print(f"MixedLM failed for {s}: {e}")
shrunk=pd.DataFrame(shrunk); shrunk.to_csv(OUTDIR/"08l_country_gradient_mixedlm_shrunken_slopes.csv",index=False)
if not shrunk.empty:
    sw=shrunk.pivot(index="country",columns="source",values="shrunk_gradient"); sc=[]
    for model in SOURCE_ORDER[1:]:
        if "Human" in sw and model in sw:
            d=sw[["Human",model]].dropna(); sc.append({"model":model,"n_countries":len(d),
                "pearson_r_shrunken":stats.pearsonr(d.Human,d[model]).statistic,
                "spearman_rho_shrunken":stats.spearmanr(d.Human,d[model]).statistic})
    shrink_corr=pd.DataFrame(sc); shrink_corr.to_csv(OUTDIR/"08m_country_gradient_mixedlm_correspondence.csv",index=False)
    display(shrink_corr.round(4))

# -------------------------------------------------------------------
# 5I. Main scatter plots + Iraq-excluded sensitivity/zoom plots
# -------------------------------------------------------------------
def draw_gradient_plot(d,model,filename,title_suffix="",hier_ci=None):
    fit=smf.ols(f'Q("{model}") ~ Human',d).fit(); pr=stats.pearsonr(d.Human,d[model]).statistic
    sr=stats.spearmanr(d.Human,d[model]).statistic
    fig,ax=plt.subplots(figsize=(11,9)); ax.scatter(d.Human,d[model],s=28,alpha=.75)
    lo=min(d.Human.min(),d[model].min()); hi=max(d.Human.max(),d[model].max()); pad=(hi-lo)*.06 if hi>lo else .01
    lo-=pad; hi+=pad
    ax.plot([lo,hi],[lo,hi],"--",linewidth=1,label="45° equality line")
    xs=np.linspace(lo,hi,100); ax.plot(xs,fit.params.Intercept+fit.params.Human*xs,linewidth=1.2,label="Cross-country fitted line")
    texts=[ax.text(r.Human,r[model],r.country_name,fontsize=7) for _,r in d.iterrows()]
    adjust_text(texts,ax=ax,arrowprops=dict(arrowstyle="-",lw=.35,alpha=.5))
    ax.axhline(0,linewidth=.6,alpha=.5); ax.axvline(0,linewidth=.6,alpha=.5); ax.set_xlim(lo,hi); ax.set_ylim(lo,hi)
    ax.set_xlabel("Human country-specific normalized income–dispersion gradient")
    ax.set_ylabel(f"{model} country-specific normalized income–dispersion gradient")
    ci=""
    if hier_ci is not None and len(hier_ci):
        z=hier_ci.iloc[0]; ci=f"; hierarchical 95% CI [{z.pearson_r_ci_low:.2f}, {z.pearson_r_ci_high:.2f}]"
    ax.set_title(f"Country gradients: Human vs {model}{title_suffix}\nPearson r={pr:.2f}{ci}; Spearman ρ={sr:.2f}; N={len(d)}")
    ax.legend(frameon=False,fontsize=8); plt.tight_layout(); fig.savefig(OUTDIR/filename,bbox_inches="tight"); plt.show()

for model in SOURCE_ORDER[1:]:
    d=grad_wide[["country","country_name","Human",model]].dropna().copy()
    draw_gradient_plot(d,model,f"fig04_country_gradient_human_vs_{SLUG[model]}.pdf",hier_ci=hier_summary[hier_summary.model.eq(model)])
    dz=d[~d.country_name.str.casefold().eq("iraq")].copy()
    if len(dz)<len(d):
        draw_gradient_plot(dz,model,f"fig04b_country_gradient_human_vs_{SLUG[model]}_EXCL_IRAQ.pdf",title_suffix=" (sensitivity: Iraq excluded)")

print("Country-gradient analyses complete.")
print("Main estimates retain all countries; Iraq exclusion is sensitivity/visualization only.")
print("Cross-country inference uses the two-stage country + respondent hierarchical bootstrap.")


## 6. Supporting RIF-variance regressions
Categorical Q288 + country FE is the main supporting specification; a linear-Q288 slope is a compact summary. SEs clustered by country.

In [ ]:
rr=[]; rl=[]
for s in SOURCE_ORDER:
    col=source_to_col[s]; d=analysis[["country","Q288",col]].rename(columns={col:"ls"}).copy()
    d["rifv"]=(d.ls-d.ls.mean())**2
    cat=smf.ols("rifv ~ C(Q288) + C(country)",d).fit(cov_type="cluster",cov_kwds={"groups":d.country})
    for q in INCOME_GROUPS:
        term=f"C(Q288)[T.{q}]"
        rr.append({"source":s,"Q288":q,"coef_vs_1":0 if q==1 else cat.params.get(term,np.nan),
                   "se":0 if q==1 else cat.bse.get(term,np.nan),
                   "p":np.nan if q==1 else cat.pvalues.get(term,np.nan)})
    lin=smf.ols("rifv ~ Q288 + C(country)",d).fit(cov_type="cluster",cov_kwds={"groups":d.country})
    rl.append({"source":s,"income_slope":lin.params["Q288"],"se":lin.bse["Q288"],"p":lin.pvalues["Q288"]})
pd.DataFrame(rr).to_csv(OUTDIR/"09_RIF_variance_categorical.csv",index=False)
riflin=pd.DataFrame(rl); riflin.to_csv(OUTDIR/"10_RIF_variance_linear_summary.csv",index=False)
display(riflin.round(5))

## 7. Supporting/prespecified adjacent-income RIF–Oaxaca diagnostic

This supporting analysis follows the diagnostic specification established in the earlier human-analysis notebook as closely as the frozen analysis file permits.

The usable adjustment set is:

- age (`Q262`)
- sex (`Q260`)
- marital status (`Q273`)
- education (`Q275`)
- number of children (`Q274`)
- country fixed effects

Self-rated health (`Q47`) and religious-person identification (`Q173`) were part of the earlier planned diagnostic control set, but both columns contain **zero valid observations** in the frozen prediction/analysis file and therefore cannot be included. No replacement covariates are added post hoc.

Because the counterfactual prediction uses a model estimated in the lower-income group to predict the adjacent higher-income group, each adjacent comparison is restricted to **common country support**: countries must contain usable observations in both income groups. This is necessary for country fixed effects to be defined in both the estimation and counterfactual samples. The common-support restriction is applied identically to humans and all six LLMs.

The decomposition is estimated separately for each adjacent pair of Q288 income groups and for humans and each LLM. It is a RIF-based diagnostic analogue rather than an exact replication of a particular Oaxaca decomposition estimator.


In [ ]:
# ============================================================
# SUPPORTING / PRESPECIFIED: ADJACENT-INCOME RIF-OAXACA DIAGNOSTIC
# ============================================================

# Keep the predetermined usable demographic controls.
# Q47 (health) and Q173 (religious-person identification) are deliberately
# omitted because they contain zero valid observations in the frozen file.
planned_controls = ["Q262", "Q260", "Q273", "Q275", "Q274"]
unusable_planned_controls = ["Q47", "Q173"]

# Audit availability rather than merely checking whether a column exists.
control_audit = []
for c in planned_controls + unusable_planned_controls:
    exists = c in analysis.columns
    n_valid = int(pd.to_numeric(analysis[c], errors="coerce").notna().sum()) if exists else 0
    control_audit.append({
        "variable": c,
        "exists": exists,
        "n_valid": n_valid,
        "pct_valid": 100 * n_valid / len(analysis)
    })

control_audit = pd.DataFrame(control_audit)
control_audit.to_csv(OUTDIR / "11a_RIF_Oaxaca_control_audit.csv", index=False)

controls = [
    c for c in planned_controls
    if c in analysis.columns
    and pd.to_numeric(analysis[c], errors="coerce").notna().sum() > 0
]

if controls != planned_controls:
    raise ValueError(
        "One or more of the five intended usable Oaxaca controls is unavailable. "
        f"Expected {planned_controls}; found {controls}. Stop and investigate."
    )

# Confirm the two previously planned variables are genuinely unusable.
for c in unusable_planned_controls:
    if c in analysis.columns:
        n_valid = pd.to_numeric(analysis[c], errors="coerce").notna().sum()
        if n_valid > 0:
            print(
                f"NOTE: {c} now has {n_valid} valid observations. "
                "The frozen-file audit should be revisited before changing the locked specification."
            )

print("RIF-Oaxaca controls used:", controls)
print("Country fixed effects: YES")
print("Not used because zero valid observations in frozen file:", unusable_planned_controls)
display(control_audit.round(2))

# Continuous vs categorical treatment follows the earlier diagnostic.
continuous_controls = ["Q262", "Q274"]  # age, number of children
terms = [
    c if c in continuous_controls else f"C({c})"
    for c in controls
] + ["C(country)"]
rhs = " + ".join(terms)

ox = []

for s in SOURCE_ORDER:
    col = source_to_col[s]

    d = analysis[["country", "Q288", col] + controls].rename(
        columns={col: "ls"}
    ).copy()

    d["ls"] = pd.to_numeric(d["ls"], errors="coerce")
    d["Q288"] = pd.to_numeric(d["Q288"], errors="coerce")
    for c in controls:
        d[c] = pd.to_numeric(d[c], errors="coerce")

    for lo in range(1, 10):
        hi = lo + 1

        pair = d[d["Q288"].isin([lo, hi])].dropna(
            subset=["ls", "Q288", "country"] + controls
        ).copy()

        # Country FE require common support: a country must occur in both
        # adjacent income groups. Without this restriction, prediction can
        # fail when the higher-income group contains a country that was absent
        # from the lower-income estimation sample (e.g. PAK for Q288 9 -> 10).
        countries_lo = set(pair.loc[pair["Q288"].eq(lo), "country"].unique())
        countries_hi = set(pair.loc[pair["Q288"].eq(hi), "country"].unique())
        common_countries = countries_lo & countries_hi

        pair = pair[pair["country"].isin(common_countries)].copy()

        g0 = pair[pair["Q288"].eq(lo)].copy()
        g1 = pair[pair["Q288"].eq(hi)].copy()

        if len(g0) < 100 or len(g1) < 100:
            ox.append({
                "source": s,
                "lower": lo,
                "higher": hi,
                "n_lower": len(g0),
                "n_higher": len(g1),
                "n_common_countries": len(common_countries),
                "status": "skipped_small_cell"
            })
            continue

        # RIF for the variance in the lower-income group.
        mu0 = g0["ls"].mean()
        g0["rifv"] = (g0["ls"] - mu0) ** 2

        try:
            m0 = smf.ols(f"rifv ~ {rhs}", data=g0).fit()

            # Counterfactual lower-group variance if it had the observed
            # covariate composition of the adjacent higher-income group.
            cf = float(m0.predict(g1).mean())

            v0 = float(g0["ls"].var(ddof=0))
            v1 = float(g1["ls"].var(ddof=0))
            total = v1 - v0
            composition = cf - v0
            structure = v1 - cf

            ox.append({
                "source": s,
                "lower": lo,
                "higher": hi,
                "n_lower": len(g0),
                "n_higher": len(g1),
                "n_common_countries": len(common_countries),
                "variance_lower": v0,
                "variance_higher": v1,
                "total_change": total,
                "composition_component": composition,
                "structure_component": structure,
                "decomposition_check": composition + structure - total,
                "status": "ok"
            })

        except Exception as e:
            ox.append({
                "source": s,
                "lower": lo,
                "higher": hi,
                "n_lower": len(g0),
                "n_higher": len(g1),
                "n_common_countries": len(common_countries),
                "status": "error",
                "error": repr(e)
            })

oaxaca = pd.DataFrame(ox)

# Guard against the previous silent-empty-output failure.
if oaxaca.empty:
    raise RuntimeError(
        "RIF-Oaxaca produced no rows. This should no longer occur after "
        "removing the two all-missing controls."
    )

n_ok = int((oaxaca["status"] == "ok").sum())
n_expected = len(SOURCE_ORDER) * 9

print(f"\nSuccessful adjacent-income decompositions: {n_ok}/{n_expected}")

if n_ok != n_expected:
    failed = oaxaca.loc[oaxaca["status"] != "ok",
                        ["source", "lower", "higher", "status", "error"]]
    print(failed.to_string(index=False))
    raise RuntimeError(
        f"RIF-Oaxaca completed {n_ok}/{n_expected} decompositions. "
        "All 63 should succeed after imposing common country support."
    )

# Numerical identity check for successful decompositions.
ok = oaxaca[oaxaca["status"] == "ok"].copy()
if not ok.empty:
    max_check = ok["decomposition_check"].abs().max()
    print(f"Maximum decomposition identity error: {max_check:.3e}")

oaxaca.to_csv(
    OUTDIR / "11_RIF_Oaxaca_adjacent_income_diagnostic.csv",
    index=False
)

display(oaxaca.head(30).round(5))


### 7A. Country-cluster bootstrap uncertainty for the RIF–Oaxaca decomposition

To quantify sampling uncertainty in the decomposition, this section resamples countries with replacement while keeping all respondents from a sampled country together. The bootstrap reproduces the common-country-support restriction and the lower-income reference specification used for the point estimates. Percentile 95% confidence intervals and bootstrap standard errors are reported for the total, composition, and residual components for all nine adjacent-income comparisons.

The implementation uses country-level sufficient statistics for speed; it is algebraically equivalent to duplicating all observations from a country each time that country is selected in a bootstrap draw.


In [ ]:
# ============================================================
# RIF-OAXACA COUNTRY-CLUSTER BOOTSTRAP UNCERTAINTY
# ============================================================

import patsy

OAX_BOOT_REPS = 200
OAX_BOOT_SEED = 20260925
rng_ox = np.random.default_rng(OAX_BOOT_SEED)

def _weighted_var_from_sums(n, sy, sy2):
    if n <= 0:
        return np.nan
    mu = sy / n
    return max(sy2 / n - mu * mu, 0.0)

def _oaxaca_bootstrap_one_source_pair(s, lo, hi):
    col = source_to_col[s]

    d = analysis[["country", "Q288", col] + controls].rename(
        columns={col: "ls"}
    ).copy()

    d["ls"] = pd.to_numeric(d["ls"], errors="coerce")
    d["Q288"] = pd.to_numeric(d["Q288"], errors="coerce")
    for c in controls:
        d[c] = pd.to_numeric(d[c], errors="coerce")

    pair = d[d["Q288"].isin([lo, hi])].dropna(
        subset=["ls", "Q288", "country"] + controls
    ).copy()

    countries_lo = set(pair.loc[pair["Q288"].eq(lo), "country"].unique())
    countries_hi = set(pair.loc[pair["Q288"].eq(hi), "country"].unique())
    common = sorted(countries_lo & countries_hi)

    pair = pair[pair["country"].isin(common)].copy()
    g0 = pair[pair["Q288"].eq(lo)].copy()
    g1 = pair[pair["Q288"].eq(hi)].copy()

    if len(g0) < 100 or len(g1) < 100 or len(common) < 2:
        return None

    # Build the exact same formula design used by the point-estimate
    # regression, but obtain Patsy's DesignInfo directly. This avoids
    # relying on statsmodels' model.data.design_info, which is not
    # available in some statsmodels versions (PandasData).
    mu0 = g0["ls"].mean()
    g0["rifv"] = (g0["ls"] - mu0) ** 2

    _, X0_df = patsy.dmatrices(
        f"rifv ~ {rhs}",
        data=g0,
        return_type="dataframe"
    )
    design_info = X0_df.design_info
    X0 = np.asarray(X0_df, dtype=float)
    X1 = np.asarray(
        patsy.build_design_matrices(
            [design_info], g1, return_type="dataframe"
        )[0],
        dtype=float
    )

    y0 = g0["ls"].to_numpy(float)
    y1 = g1["ls"].to_numpy(float)
    p = X0.shape[1]
    C = len(common)
    cmap = {c:i for i,c in enumerate(common)}

    # Country-level sufficient statistics for the lower-group regression
    # and for lower/higher-group means and variances.
    XtX = np.zeros((C, p, p))
    Xt1 = np.zeros((C, p))
    Xty = np.zeros((C, p))
    Xty2 = np.zeros((C, p))
    sumX1 = np.zeros((C, p))

    n0 = np.zeros(C); sy0 = np.zeros(C); sy20 = np.zeros(C)
    n1 = np.zeros(C); sy1 = np.zeros(C); sy21 = np.zeros(C)

    c0 = g0["country"].to_numpy()
    c1 = g1["country"].to_numpy()

    for c in common:
        k = cmap[c]
        i0 = np.flatnonzero(c0 == c)
        i1 = np.flatnonzero(c1 == c)

        X0c, y0c = X0[i0], y0[i0]
        X1c, y1c = X1[i1], y1[i1]

        XtX[k] = X0c.T @ X0c
        Xt1[k] = X0c.sum(axis=0)
        Xty[k] = X0c.T @ y0c
        Xty2[k] = X0c.T @ (y0c ** 2)
        sumX1[k] = X1c.sum(axis=0)

        n0[k] = len(y0c)
        sy0[k] = y0c.sum()
        sy20[k] = np.sum(y0c ** 2)

        n1[k] = len(y1c)
        sy1[k] = y1c.sum()
        sy21[k] = np.sum(y1c ** 2)

    boot = np.full((OAX_BOOT_REPS, 3), np.nan)

    for b in range(OAX_BOOT_REPS):
        # Resample countries with replacement. Multiplicity is equivalent
        # to duplicating each sampled country's respondent block.
        mult = rng_ox.multinomial(C, np.repeat(1.0 / C, C)).astype(float)

        N0 = mult @ n0
        N1 = mult @ n1
        if N0 <= 0 or N1 <= 0:
            continue

        SY0 = mult @ sy0
        SY20 = mult @ sy20
        SY1 = mult @ sy1
        SY21 = mult @ sy21

        mu0b = SY0 / N0

        XX = np.tensordot(mult, XtX, axes=(0, 0))
        # X'[(y-mu)^2] = X'y^2 - 2mu X'y + mu^2 X'1
        Xrif = (
            mult @ Xty2
            - 2.0 * mu0b * (mult @ Xty)
            + (mu0b ** 2) * (mult @ Xt1)
        )

        beta = np.linalg.pinv(XX) @ Xrif

        x1bar = (mult @ sumX1) / N1
        cf = float(x1bar @ beta)

        v0 = _weighted_var_from_sums(N0, SY0, SY20)
        v1 = _weighted_var_from_sums(N1, SY1, SY21)

        total = v1 - v0
        composition = cf - v0
        residual = v1 - cf
        boot[b] = [total, composition, residual]

    return boot

ci_rows = []

for s in SOURCE_ORDER:
    for lo in range(1, 10):
        hi = lo + 1

        boot = _oaxaca_bootstrap_one_source_pair(s, lo, hi)
        if boot is None:
            continue

        point_row = oaxaca[
            oaxaca["source"].eq(s)
            & oaxaca["lower"].eq(lo)
            & oaxaca["higher"].eq(hi)
            & oaxaca["status"].eq("ok")
        ].iloc[0]

        point = {
            "total": float(point_row["total_change"]),
            "composition": float(point_row["composition_component"]),
            "residual": float(point_row["structure_component"])
        }

        for k, component in enumerate(["total", "composition", "residual"]):
            vals = boot[:, k]
            vals = vals[np.isfinite(vals)]
            lo_ci, hi_ci = np.quantile(vals, [0.025, 0.975])
            ci_rows.append({
                "source": s,
                "lower": lo,
                "higher": hi,
                "component": component,
                "estimate": point[component],
                "bootstrap_se": float(np.std(vals, ddof=1)),
                "ci_low": float(lo_ci),
                "ci_high": float(hi_ci),
                "bootstrap_reps_requested": OAX_BOOT_REPS,
                "bootstrap_reps_valid": len(vals),
                "bootstrap_level": 0.95,
                "bootstrap_unit": "country"
            })

oaxaca_ci_long = pd.DataFrame(ci_rows)
oaxaca_ci_long.to_csv(
    OUTDIR / "11b_RIF_Oaxaca_country_cluster_bootstrap_CI_long.csv",
    index=False
)

# Wide manuscript/SI-friendly table.
oaxaca_ci_wide = (
    oaxaca_ci_long
    .pivot_table(
        index=["source", "lower", "higher"],
        columns="component",
        values=["estimate", "bootstrap_se", "ci_low", "ci_high"]
    )
)
oaxaca_ci_wide.columns = [
    f"{component}_{stat}" for stat, component in oaxaca_ci_wide.columns
]
oaxaca_ci_wide = oaxaca_ci_wide.reset_index()

oaxaca_ci_wide.to_csv(
    OUTDIR / "11c_RIF_Oaxaca_country_cluster_bootstrap_CI_wide.csv",
    index=False
)

# Main-text transitions for quick inspection.
main_oax_ci = oaxaca_ci_wide[
    ((oaxaca_ci_wide["lower"] == 1) & (oaxaca_ci_wide["higher"] == 2))
    | ((oaxaca_ci_wide["lower"] == 9) & (oaxaca_ci_wide["higher"] == 10))
].copy()

main_oax_ci.to_csv(
    OUTDIR / "11d_RIF_Oaxaca_main_transitions_with_CI.csv",
    index=False
)

print(
    f"RIF-Oaxaca uncertainty complete: {OAX_BOOT_REPS} country-cluster "
    "bootstrap draws per source × adjacent-income comparison."
)
display(main_oax_ci.round(4))


## 8. EXPLORATORY: distributional anatomy
Complete 1–10 distributions, lower/upper tails, and empirical unconditional quantiles. Because LS is discrete, this notebook does not impose a continuous-density UQR approximation.

In [ ]:
dist=long.groupby(["source","Q288","ls"],observed=True).size().rename("n").reset_index()
dist["share"]=dist.n/dist.groupby(["source","Q288"],observed=True).n.transform("sum")
dist.to_csv(OUTDIR/"12_score_distribution_by_income_source.csv",index=False)
tails=mom[["source","Q288","misery3","low7","high8","max10","p10","p25","p50","p75","p90"]].copy()
tails.to_csv(OUTDIR/"13_exploratory_tails_quantiles.csv",index=False)

for metric,ylabel,fname in [("misery3","Share with LS ≤ 3","fig05_misery3.pdf"),
                             ("high8","Share with LS ≥ 8","fig06_high8.pdf")]:
    plt.figure(figsize=(9,6))
    for s in SOURCE_ORDER:
        g=tails[tails.source.eq(s)]; plt.plot(g.Q288,g[metric],marker="o",label=s)
    plt.xlabel("WVS income-group position (Q288)"); plt.ylabel(ylabel); plt.xticks(INCOME_GROUPS)
    plt.legend(fontsize=8); plt.tight_layout(); plt.savefig(OUTDIR/fname,bbox_inches="tight"); plt.show()

## 9. EXPLORATORY: individual prediction error
Compare model MAE with the error from predicting the human conditional median within each income group.

In [ ]:
med=analysis.groupby("Q288").Q49.median()
analysis["median_pred"]=analysis.Q288.map(med)
analysis["baseline_ae"]=(analysis.Q49-analysis.median_pred).abs()
mr=[]
for model in SOURCE_ORDER[1:]:
    col=source_to_col[model]; ae=(analysis[col]-analysis.Q49).abs()
    for q in INCOME_GROUPS:
        mask=analysis.Q288.eq(q); ma=ae[mask].mean(); ba=analysis.loc[mask,"baseline_ae"].mean()
        mr.append({"model":model,"Q288":q,"n":mask.sum(),"model_mae":ma,
                   "human_conditional_median_mae":ba,"mae_difference":ma-ba,"mae_ratio":ma/ba})
mae=pd.DataFrame(mr); mae.to_csv(OUTDIR/"14_exploratory_MAE_by_income.csv",index=False)
display(mae.round(4))

## 10. ROBUSTNESS: alternative inequality measures
Normalize variance, CV, Gini and IQR with the same fixed human income weights. SD remains primary.

In [ ]:
rob=[]
for metric in ["variance_ls","cv_ls","gini_ls","iqr_ls"]:
    for s in SOURCE_ORDER:
        g=mom.loc[mom.source.eq(s)].set_index("Q288").reindex(INCOME_GROUPS)
        vals=g[metric].values.astype(float); sc=float(np.sum(w*vals))
        for j,q in enumerate(INCOME_GROUPS):
            rob.append({"metric":metric,"source":s,"Q288":q,"raw_value":vals[j],
                        "scale":sc,"normalized_value":vals[j]/sc if sc else np.nan})
robust=pd.DataFrame(rob)
robust.to_csv(OUTDIR/"15_robustness_alternative_inequality_profiles.csv",index=False)
display(robust.head(30).round(5))

## 11. Final inventory
Interpret in this order: audit → human benchmark → mean benchmark → general compression → **primary normalized profile** → supporting analyses → exploratory analyses → robustness.

**Country-structure diagnostics (supporting/post-lock)**
- `07_within_country_FE_slopes.csv`: conventional country-FE slopes with country-clustered SEs.
- `08a_country_FE_normalized_sd_slopes.csv` and `08a2_country_FE_normalized_sd_categorical.csv`: normalized country-FE analyses.
- `08c_country_specific_normalized_sd_gradients.csv` and `08d_country_gradient_wide.csv`: country-specific normalized gradients.
- `08e_country_gradient_correspondence_descriptive.csv`: full-sample Pearson/Spearman correspondence, fitted slope, R², sign agreement, and 45°-line distances.
- `08f_leave_one_country_out.csv`, `08f2_country_influence_diagnostics.csv`, `08f3_excluding_iraq_sensitivity.csv`, `08f4_leave_one_country_out_summary.csv`: influence and outlier sensitivity.
- `08g_country_gradient_respondent_bootstrap_CI.csv`: respondent-bootstrap CIs for individual country gradients.
- `08h_hierarchical_country_respondent_bootstrap_replicates.csv` and `08i_country_gradient_correspondence_HIERARCHICAL_bootstrap_CI.csv`: two-stage hierarchical-bootstrap inference for cross-country correspondence.
- `08j_country_gradient_reliability.csv` and `08k_country_gradient_disattenuated_correlation_sensitivity.csv`: approximate errors-in-variables sensitivity.
- `08l_country_gradient_mixedlm_shrunken_slopes.csv` and `08m_country_gradient_mixedlm_correspondence.csv`: random-slope shrinkage robustness.
- `fig04_country_gradient_human_vs_*.png`: six main all-country plots with identical x/y scales and hierarchical-bootstrap CI.
- `fig04b_country_gradient_human_vs_*_EXCL_IRAQ.png`: six Iraq-excluded sensitivity/zoom plots.

**Inference distinction**
- Section 4: paired **country-cluster bootstrap** for the aggregate primary confirmatory profile.
- Section 5 country-specific CIs: paired **respondent bootstrap within countries**.
- Section 5 cross-country correspondence: **two-stage hierarchical bootstrap**, resampling countries and then respondents within selected countries.

In [ ]:

print("FIGURES GENERATED BY THIS RUN (PDF only):")
for p in sorted(OUTDIR.glob("fig*.pdf")):
    print(" -", p.name)

print("\nTABULAR / NUMERICAL OUTPUTS:")
for p in sorted(OUTDIR.glob("*.csv")):
    print(" -", p.name)


In [ ]:
# ============================================================
# ZIP ALL RESULTS AND DOWNLOAD DIRECTLY TO YOUR COMPUTER
# ============================================================

from pathlib import Path
import shutil
import base64
from IPython.display import HTML, display

results_dir = Path("output/full_wvs/analysis_locked")
zip_base = Path("output/full_wvs/WVS_LLM_analysis_results")
zip_file = zip_base.with_suffix(".zip")

# Remove old ZIP
if zip_file.exists():
    zip_file.unlink()

# Create fresh ZIP containing all current results
shutil.make_archive(
    str(zip_base),
    "zip",
    root_dir=results_dir
)

size_mb = zip_file.stat().st_size / (1024**2)
print(f"Created: {zip_file}")
print(f"Size: {size_mb:.1f} MB")

# Read ZIP and encode it for direct browser download
with open(zip_file, "rb") as f:
    encoded = base64.b64encode(f.read()).decode()

filename = zip_file.name

# Create a download button and automatically click it
html = f"""
<a id="download-results"
   download="{filename}"
   href="data:application/zip;base64,{encoded}"
   style="
       display:inline-block;
       padding:12px 20px;
       background:#2563eb;
       color:white;
       text-decoration:none;
       border-radius:6px;
       font-weight:600;
       margin-top:10px;
   ">
   Download all Notebook 4 results
</a>

<script>
setTimeout(function() {{
    document.getElementById("download-results").click();
}}, 500);
</script>
"""

display(HTML(html))

<a style='text-decoration:none;line-height:16px;display:flex;color:#5B5B62;padding:10px;justify-content:end;' href='https://deepnote.com?utm_source=created-in-deepnote-cell&projectId=7aaa7215-b731-433d-9b62-8be4a70a4410' target="_blank">

Created in <span style='font-weight:600;margin-left:4px;'>Deepnote</span></a>